Testing things for my python script

In [3]:
import geopandas as gpd
import matplotlib.pyplot as plt

In [4]:
# Loading data
# The precinct data is from 2026 and the polls data is from 2020 so there are some errors.
# Source: https://www.nconemap.gov/datasets/nconemap::north-carolina-voting-precincts/about 
all_precincts = gpd.read_file("Voting_Precincts.geojson")
# Source: https://www.nconemap.gov/datasets/nconemap::north-carolina-voting-polling-places/about 
polls = gpd.read_file("North_Carolina_Voting_Polling_Places.geojson")
# May not need the county data, delete later if not. 
# counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_500k.zip")

In [5]:
# Selecting only the columns needed from each dataframe. 
polls = polls[["USER_pol_1", "USER_addre", "USER_city", "USER_zip","geometry"]]
all_precincts = all_precincts[["prec_id", "county_nam", "geometry"]]

# And renaming 
polls = polls.rename(columns={"USER_pol_1":"poll_name", "USER_addre":"address", "USER_city":"city",
                       "USER_zip":"zip"})
all_precincts = all_precincts.rename(columns={"county_nam":"county"})

# Formatting the county column 
all_precincts["county"] = all_precincts["county"].str.capitalize()

# Selecting just the precincts for one county. 
# Change "Wake" into an argparse argument later. 
precincts = all_precincts[all_precincts["county"] == "Alamance"].copy()

In [6]:
# Pulling out just NC counties. 
# ncc = counties[counties["STATEFP"] == "37"]

# Changing the CRS
precincts = precincts.to_crs(epsg=6543)
polls = polls.to_crs(epsg=6543)
# ncc = ncc.to_crs(epsg=6543)

In [7]:
# Delete this in the script.
# fig, ax = plt.subplots()
# wake = precincts[precincts["county_nam"] == "WAKE"]
# wake_polls = gpd.sjoin(polls, wake, predicate="within")
# wake.plot(ax=ax, edgecolor="#000000", facecolor="white", lw=1)
# wake_polls.plot(ax=ax, markersize=3, color="blue")

Process
Let the user pick one county 
Join the polls to the precincts 
Then select the precincts that match the county the user inputs
Iterate through each precinct to show on the map 
for precinct in prec_id:
    make a page 

In [8]:
# Joining the precinct data to the polls. 
polls_join = gpd.sjoin(polls, precincts, predicate="intersects")
polls_join = polls_join.drop(columns="index_right")

In [9]:
# Creating a function with name as an argument.
# Will later iterate through filling in prec_id.

# Creating function for county map with the precinct highlighted. 
def highlight(name, ax): 
    selected = precincts[precincts["prec_id"] == name]
    precincts.plot(ax=ax, edgecolor="grey", facecolor="white")
    selected.plot(ax=ax, edgecolor="#000000", facecolor="#017aca", lw=1.5)
    ax.set_axis_off()

In [10]:
# Map with just precinct.
def precinct_map(name, ax):
    selected = precincts[precincts["prec_id"] == name]
    selected.plot(ax=ax, edgecolor="#000000", facecolor="#017aca", lw=1.5)

    # Plotting polling places 
    precinct_poll = polls_join[polls_join["prec_id"] == name]
    if len(precinct_poll) > 0:
        precinct_poll.plot(ax=ax, color="#FFC400", markersize=60, marker="*")

    ax.set_axis_off()

In [11]:
def summary(name):
    row = polls_join[polls_join["prec_id"] == name]
    if len(row) > 0: 
        poll_address = row["address"].iloc[0]
        text = f"Poll address: {poll_address}"
    else: 
        text = "No poll"
    return text 

In [12]:
# A function to create each page by combining the elements.
def page(precinct, pdf):
    fig, axes = plt.subplots(
        2, 1,
        figsize=(8,10),
        gridspec_kw={"height_ratios": [3, 1]}
    )

    precinct_map(precinct, axes[0])
    highlight(precinct, axes[1])

    fig.suptitle(f"Precinct {precinct}", 
                 fontname="Times New Roman", 
                 ha="center", va="center", 
                 fontsize=18, 
                 fontweight="bold")
    fig.text(0.5, 0.93, 
             summary(precinct), 
             fontname="Times New Roman", 
             ha="center", 
             va="center", 
             fontsize=12)

    pdf.savefig(fig)
    plt.close(fig)

In [13]:
from matplotlib.backends.backend_pdf import PdfPages 
with PdfPages("county_polls.pdf") as pdf: 
    for prec in precincts["prec_id"]:
        page(prec, pdf)